# EasyVisa Project: Visa Approval Prediction

## 1. Exploratory Data Analysis (EDA)

### Problem Definition

The Immigration and Nationality Act (INA) of the US permits foreign workers to come to the United States to work on either a temporary or permanent basis. The process of reviewing every case is becoming a tedious task as the number of applicants is increasing every year.

The objective is to develop a Machine Learning based solution that can help the Office of Foreign Labor Certification (OFLC) in shortlisting the candidates having higher chances of VISA approval. As a data scientist at EasyVisa, the goal is to:
1. Facilitate the process of visa approvals.
2. Recommend a suitable profile for the applicants for whom the visa should be certified or denied based on the drivers that significantly influence the case status.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score, recall_score, precision_score, make_scorer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, AdaBoostClassifier, GradientBoostingClassifier, StackingClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 8]

# Load the dataset
data = pd.read_csv('EasyVisa.csv')

# Display the first few rows
data.head()

### Observations after loading data:
- The dataset contains multiple features related to both the employee and the employer.
- The target variable is `case_status`, indicating if the visa was 'Certified' or 'Denied'.

In [ ]:
# Function to compute performance metrics
def model_performance_classification(model, predictors, target):
    """
    Function to compute different metrics to check classification model performance
    """
    pred = model.predict(predictors)
    acc = accuracy_score(target, pred)
    recall = recall_score(target, pred)
    precision = precision_score(target, pred)
    f1 = f1_score(target, pred)

    df_perf = pd.DataFrame(
        {"Accuracy": acc, "Recall": recall, "Precision": precision, "F1": f1},
        index=[0],
    )
    return df_perf

def confusion_matrix_plot(model, predictors, target):
    """
    To plot the confusion_matrix with percentages
    """
    y_pred = model.predict(predictors)
    cm = confusion_matrix(target, y_pred)
    labels = np.asarray(
        [["{0:0.0f}".format(item), "{0:.2%}".format(item / cm.flatten().sum())] for item in cm.flatten()]
    ).reshape(2, 2, 2)

    label_map = np.array(["True Neg", "False Pos", "False Neg", "True Pos"]).reshape(2, 2)
    annot = np.array(
        ["{}\n{}\n{}".format(l1, l2, l3) for l1, l2, l3 in zip(label_map.flatten(), labels[:, :, 0].flatten(), labels[:, :, 1].flatten())]
    ).reshape(2, 2)

    plt.figure(figsize=(8, 5))
    sns.heatmap(cm, annot=annot, fmt="", cbar=False)
    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.show()

def plot_feature_importance(model, predictors):
    """
    To plot the feature importance of a model
    """
    importances = model.feature_importances_
    indices = np.argsort(importances)
    feature_names = predictors.columns
    
    plt.figure(figsize=(12, 12))
    plt.title('Feature Importances')
    plt.barh(range(len(indices)), importances[indices], color='violet', align='center')
    plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
    plt.xlabel('Relative Importance')
    plt.show()

### Observations on Functions:
- Added a utility function `plot_feature_importance` to visualize which drivers most significantly influence visa approval status, matching the reference standards.

## 2. Data Pre-processing

In [ ]:
df = data.copy()
df['no_of_employees'] = df['no_of_employees'].abs()
df.drop('case_id', axis=1, inplace=True)

def normalize_wage(row):
    if row['unit_of_wage'] == 'Hour': return row['prevailing_wage'] * 40 * 52
    if row['unit_of_wage'] == 'Week': return row['prevailing_wage'] * 52
    if row['unit_of_wage'] == 'Month': return row['prevailing_wage'] * 12
    return row['prevailing_wage']

df['prevailing_wage'] = df.apply(normalize_wage, axis=1)
df.drop('unit_of_wage', axis=1, inplace=True)

df['case_status'] = df['case_status'].map({'Certified': 1, 'Denied': 0})
df = pd.get_dummies(df, drop_first=True)

X = df.drop('case_status', axis=1)
y = df['case_status']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

## 3. Model Building - Original Data

### AdaBoost Classifier

In [ ]:
ab = AdaBoostClassifier(random_state=1)
ab.fit(X_train, y_train)

print("AdaBoost Performance (Test):")
display(model_performance_classification(ab, X_test, y_test))
plot_feature_importance(ab, X_train)

### Observations on AdaBoost Feature Importance:
- AdaBoost heavily relies on specific threshold values in numerical features like `prevailing_wage` and `no_of_employees` to make its decisions.

### Gradient Boosting Classifier

In [ ]:
gb = GradientBoostingClassifier(random_state=1)
gb.fit(X_train, y_train)

print("Gradient Boosting Performance (Test):")
display(model_performance_classification(gb, X_test, y_test))
plot_feature_importance(gb, X_train)

### Observations on Gradient Boosting Feature Importance:
- Unlike AdaBoost, Gradient Boosting shows a more distributed importance across features, with `education` and `prevailing_wage` often being the top drivers.

### XGBoost Classifier

In [ ]:
xgb = XGBClassifier(random_state=1, eval_metric='logloss')
xgb.fit(X_train, y_train)

print("XGBoost Performance (Test):")
display(model_performance_classification(xgb, X_test, y_test))
plot_feature_importance(xgb, X_train)

### Observations on XGBoost Feature Importance:
- XGBoost effectively captures the non-linear relationship of the drivers, with `education_of_employee` significantly impacting the certification likelihood.

## 7. Model Comparison and Final Selection

In [ ]:
models = [ab, gb, xgb]
names = ["AdaBoost", "Gradient Boosting", "XGBoost"]

results = []
for model, name in zip(models, names):
    results.append(model_performance_classification(model, X_test, y_test))

comparison_df = pd.concat(results)
comparison_df.index = names
comparison_df.sort_values(by="F1", ascending=False)

## 8. Actionable Insights & Recommendations

### Insights:
- Feature importance plots confirm that employee credentials and wages are the primary drivers for visa certification.

### Recommendations:
- Standardize the review process for applicants falling into the high-importance feature categories identified by the models.